In [ ]:
# Shared setup for CAS presence visualizations
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DOCUMENT_REGISTRY = PROJECT_ROOT / "data" / "metadata" / "registries" / "document_registry.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "images" / "CAS"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESENCE_COLORS = {
    "CAS present": "#1D9E75",
    "No CAS": "#9AA0A6",
}


def load_cas_presence_data(year: str | int | None = None) -> pd.DataFrame:
    documents = pd.read_csv(DOCUMENT_REGISTRY, dtype=str)
    df = documents[
        documents["doc_type"].eq("Scientific study")
        & documents["has_CAS"].notna()
    ].copy()
    if year is not None:
        df = df[df["publication_year"].eq(str(year))].copy()
    return df.reset_index(drop=True)


def save_cas_presence_graph(df: pd.DataFrame, title: str, outfile: str) -> Path:
    counts = pd.Series(
        {
            "CAS present": int(df["has_CAS"].eq("True").sum()),
            "No CAS": int(df["has_CAS"].eq("False").sum()),
        }
    )
    total = int(counts.sum())

    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    bars = ax.bar(counts.index, counts.values, color=[PRESENCE_COLORS[label] for label in counts.index], width=0.62)
    ax.set_ylabel("Scientific studies")
    ax.set_title(title, loc="left", fontsize=13, fontweight="600")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(counts.max() * 1.18, 1))

    for bar, value in zip(bars, counts.values):
        pct = (value / total * 100) if total else 0
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + max(counts.max(), 1) * 0.025,
            f"{value:,}\n{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    ax.text(0, -0.18, f"Analyzed scientific studies with non-missing has_CAS: n={total:,}", transform=ax.transAxes, fontsize=9, color="#555")
    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.12, right=0.98)
    path = OUTPUT_DIR / outfile
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


In [ ]:
# Visualization: CAS presence in 2024 scientific studies
df_cas_2024 = load_cas_presence_data(2024)
path = save_cas_presence_graph(
    df_cas_2024,
    "CAS presence in 2024 scientific studies",
    "CAS_presence_2024_scientific_studies.png",
)
print(path)
print(df_cas_2024["has_CAS"].value_counts().to_string())


In [ ]:
# Visualization: CAS presence in 2025 scientific studies
df_cas_2025 = load_cas_presence_data(2025)
path = save_cas_presence_graph(
    df_cas_2025,
    "CAS presence in 2025 scientific studies",
    "CAS_presence_2025_scientific_studies.png",
)
print(path)
print(df_cas_2025["has_CAS"].value_counts().to_string())
